# Day 4 / 7 — Actor Network (Policy π)
### `src/models/actor.py` — Macro Graph Prior (GNN) skipped for now

**Updated roadmap** (Day 4 originally covered the GNN — skipped since you don't have macro/graph data yet; revised order below):

| Day | Focus |
|---|---|
| 1 ✅ | Data & windowing |
| 2 ✅ | Regime Encoder — VSN + LSTM |
| 3 ✅ | Temporal Attention (MHA) |
| ~~4~~ | ~~Macro Graph Prior (GNN/GAT)~~ — **skipped, revisit once you collect macro data** |
| **4 (today)** | **Actor Network** — hidden dims, dropout, long-only vs. long-short |
| 5 | Critic + Reward shaping — Sharpe window, EVaR penalty, transaction cost |
| 6 | PPO training loop — clip, epochs, batch size, gamma, GAE lambda — full ablation |

**Carried forward:**
```python
# Day 1
window_size = 30, n_assets = 10, all features on, normalize_method = 'zscore'
# Day 2
vsn_hidden_dim = 64, lstm_hidden_dim = 32, lstm_num_layers = 3, dropout = 0.1
# Day 3
mha_num_heads = 8, ffn_dim = 512, mha_dropout = 0.3
```

**Important simplification for today:** in your full architecture, `h_temp` (fed to the Actor) is the *fusion* of the Temporal Attention output **and** the Macro Graph Prior's graph embedding (`fusion_input_dim = lstm_hidden_dim + gnn_hidden_dim`, see `regime_encoder.py`). Since the GNN is skipped, we feed the Temporal Attention output directly into the Actor as a stand-in for `h_temp` (dimension = `lstm_hidden_dim` = 32). This is **not** a change to `actor.py` itself — it's simply what we plug into its `input_dim`. When you add the GNN back later, `input_dim` becomes `latent_state_dim` (128 by default) instead.

**How we'll actually train it:** the real reward (Sharpe − costs − EVaR penalty, PPO's clipped objective) is Day 5–6's territory. For today, we use a simpler, fully differentiable proxy: take the Actor's **deterministic** action (portfolio weights), compute the **realized next-day portfolio return** using those weights against real next-day per-asset returns, and directly maximize it via gradient descent (`loss = -mean(portfolio_return)`). This lets the Actor's own hyperparameters (depth, dropout, long-only vs. long-short) show a clear, interpretable effect without needing the full PPO machinery yet.


## Step 0 — Setup (same zip as before)

In [ ]:
from google.colab import files
import os

if not os.path.exists('/content/Regime detection portfolio.zip') and not os.path.exists('/content/project'):
    uploaded = files.upload()


In [ ]:
import os, zipfile, glob

ZIP_CANDIDATES = glob.glob('/content/*.zip')
PROJECT_DIR = '/content/project'

if not os.path.exists(PROJECT_DIR):
    assert ZIP_CANDIDATES, "No zip found in /content — upload it first."
    with zipfile.ZipFile(ZIP_CANDIDATES[0], 'r') as z:
        z.extractall(PROJECT_DIR)

ROOT = None
for dirpath, _, filenames in os.walk(PROJECT_DIR):
    if 'main.py' in filenames:
        ROOT = dirpath
        break
assert ROOT is not None
print("Project root:", ROOT)


In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import sys
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.config import get_config, DATA_RAW_DIR
from src.data.data_loader import CSEDataLoader
from src.data.preprocessor import Preprocessor
from src.models.vsn_lstm import VSNLSTM
from src.models.temporal_attention import TemporalAttentionBlock
from src.models.actor import ActorNetwork

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


## Step 1 — Rebuild windows + a per-asset next-day-return target

Unlike Days 2–3 (which predicted the *average* market return), the Actor needs a **per-asset** target, since it allocates weights across all 10 stocks individually.


In [ ]:
DAY1 = dict(window_size=30, normalize_method='zscore', n_assets=10)
DAY2 = dict(vsn_hidden_dim=64, lstm_hidden_dim=32, lstm_num_layers=3, dropout=0.1)
DAY3 = dict(mha_num_heads=8, ffn_dim=512, mha_dropout=0.3)

loader = CSEDataLoader(raw_data_dir=DATA_RAW_DIR)
prices, volumes = loader.load_prices_and_volumes(n_assets=DAY1['n_assets'])
asset_names = list(prices.columns)
n_assets = len(asset_names)

preprocessor = Preprocessor(window_size=DAY1['window_size'], normalize_method=DAY1['normalize_method'])
features = preprocessor.engineer_features(prices, volumes)
normalized = preprocessor.fit_normalize(features)
windows = preprocessor.create_windows(normalized.values)

# Per-asset log returns, shifted so row d holds the return realized the day AFTER d
per_asset_returns = np.log(prices / prices.shift(1))
target_next = per_asset_returns.shift(-1)

window_dates = features.index[DAY1['window_size'] - 1:]
targets = target_next.reindex(window_dates)

valid_mask = targets.notna().all(axis=1).values
windows_valid = windows[valid_mask]
targets_valid = targets.values[valid_mask].astype(np.float32)   # (N, n_assets)

n = windows_valid.shape[0]
n_train = int(n * 0.8)
X_train, X_val = windows_valid[:n_train], windows_valid[n_train:]
Y_train, Y_val = targets_valid[:n_train], targets_valid[n_train:]
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | n_assets: {n_assets}")

# Equal-weight benchmark on validation set (for later comparison)
equal_weight_val_returns = Y_val.mean(axis=1)
print(f"Equal-weight val return: mean={equal_weight_val_returns.mean():.5f}, std={equal_weight_val_returns.std():.5f}")


## Step 2 — Training probe: encoder (Days 2+3, frozen hyperparams but trainable weights) → Actor → realized return

In [ ]:
def train_probe_actor(hidden_dims=[256, 128], dropout=0.1, allow_short=False,
                       epochs=40, lr=1e-3, batch_size=32, seed=42, verbose=False):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n_features = X_train.shape[2]
    d_model = DAY2['lstm_hidden_dim']

    encoder = VSNLSTM(
        n_features=n_features,
        vsn_hidden_dim=DAY2['vsn_hidden_dim'],
        lstm_hidden_dim=d_model,
        lstm_num_layers=DAY2['lstm_num_layers'],
        dropout=DAY2['dropout'],
    ).to(device)

    attn_block = TemporalAttentionBlock(
        d_model=d_model,
        num_heads=DAY3['mha_num_heads'],
        ffn_dim=DAY3['ffn_dim'],
        dropout=DAY3['mha_dropout'],
    ).to(device)

    actor = ActorNetwork(
        input_dim=d_model,          # stand-in for h_temp (no GNN fusion today)
        n_assets=n_assets,
        hidden_dims=hidden_dims,
        dropout=dropout,
        allow_short=allow_short,
        n_regimes=3,                # unused since regime_probs=None below
    ).to(device)

    params = list(encoder.parameters()) + list(attn_block.parameters()) + list(actor.parameters())
    n_params = sum(p.numel() for p in params)
    optimizer = torch.optim.Adam(params, lr=lr)

    X_train_t = torch.FloatTensor(X_train)
    Y_train_t = torch.FloatTensor(Y_train)
    X_val_t = torch.FloatTensor(X_val).to(device)
    Y_val_t = torch.FloatTensor(Y_val).to(device)

    n_train_samples = X_train_t.shape[0]
    train_returns, val_returns = [], []

    def forward_actor(xb, deterministic):
        lstm_out, _, _ = encoder(xb)
        attn_out, _ = attn_block(lstm_out)
        h_temp_proxy = attn_out[:, -1, :]              # stand-in for h_temp
        action, log_prob, entropy = actor(h_temp_proxy, regime_probs=None, deterministic=deterministic)
        return action

    for epoch in range(epochs):
        encoder.train(); attn_block.train(); actor.train()
        perm = torch.randperm(n_train_samples)
        epoch_return = 0.0
        for i in range(0, n_train_samples, batch_size):
            idx = perm[i:i+batch_size]
            xb = X_train_t[idx].to(device)
            yb = Y_train_t[idx].to(device)

            action = forward_actor(xb, deterministic=True)   # smooth, differentiable weights
            portfolio_return = (action * yb).sum(dim=-1)      # (batch,)
            loss = -portfolio_return.mean()                   # maximize realized return

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_return += portfolio_return.sum().item()

        train_returns.append(epoch_return / n_train_samples)

        encoder.eval(); attn_block.eval(); actor.eval()
        with torch.no_grad():
            action_val = forward_actor(X_val_t, deterministic=True)
            portfolio_return_val = (action_val * Y_val_t).sum(dim=-1)
            val_returns.append(portfolio_return_val.mean().item())

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:3d} | train ret {train_returns[-1]:.6f} | val ret {val_returns[-1]:.6f}")

    # Final validation diagnostics
    with torch.no_grad():
        final_action = forward_actor(X_val_t, deterministic=True)
        final_returns_series = (final_action * Y_val_t).sum(dim=-1).cpu().numpy()
        mean_weights = final_action.mean(dim=0).cpu().numpy()
        max_weight = final_action.max(dim=-1).values.mean().item()
        weight_entropy = -(final_action.clamp_min(1e-8) * final_action.clamp_min(1e-8).log()).sum(dim=-1).mean().item()
        effective_n_assets = float(np.exp(weight_entropy))
        sharpe_like = final_returns_series.mean() / (final_returns_series.std() + 1e-8)

    return {
        'train_returns': train_returns,
        'val_returns': val_returns,
        'final_val_return': val_returns[-1],
        'n_params': n_params,
        'mean_weights': mean_weights,
        'max_weight': max_weight,
        'effective_n_assets': effective_n_assets,
        'sharpe_like': sharpe_like,
        'val_return_series': final_returns_series,
    }

print("train_probe_actor() ready.")


## Step 3 — Baseline run (`hidden_dims=[256,128]`, `dropout=0.1`, `allow_short=False`, i.e. long-only)

In [ ]:
baseline = train_probe_actor(hidden_dims=[256, 128], dropout=0.1, allow_short=False, epochs=40, verbose=True)

plt.figure(figsize=(7,4))
plt.plot(baseline['train_returns'], label='train (mean daily return)')
plt.plot(baseline['val_returns'], label='val (mean daily return)')
plt.axhline(0, color='grey', lw=0.8)
plt.xlabel('epoch'); plt.ylabel('mean portfolio return'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Baseline Actor training')
plt.show()

print(f"Params: {baseline['n_params']:,} | Final val return: {baseline['final_val_return']:.6f}")
print(f"Effective # assets used (diversification): {baseline['effective_n_assets']:.2f} / {n_assets}")
print(f"Sharpe-like ratio (val): {baseline['sharpe_like']:.4f}")


In [ ]:
# Learned average portfolio allocation vs. equal weight
x = np.arange(n_assets)
width = 0.35
plt.figure(figsize=(9,4))
plt.bar(x - width/2, baseline['mean_weights'], width, label='learned (baseline)')
plt.bar(x + width/2, [1/n_assets]*n_assets, width, label='equal-weight')
plt.xticks(x, asset_names)
plt.ylabel('weight'); plt.legend(); plt.title('Learned vs. equal-weight allocation')
plt.show()


In [ ]:
# Cumulative return: learned policy vs. equal-weight benchmark, over the validation period
cum_learned = np.cumsum(baseline['val_return_series'])
cum_equal = np.cumsum(equal_weight_val_returns)

plt.figure(figsize=(9,4))
plt.plot(cum_learned, label='learned Actor (baseline)')
plt.plot(cum_equal, label='equal-weight benchmark')
plt.xlabel('validation day'); plt.ylabel('cumulative log return'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Cumulative return: learned policy vs. equal-weight')
plt.show()


**Your task:** does the learned policy beat equal-weight on the validation window? Remember — this is trained on a *simplified* return-maximization objective (no transaction costs, no risk penalty), so don't read too much into absolute performance yet; that's what Day 5's real Sharpe/EVaR reward is for. Just note whether it concentrates into a few stocks or stays diversified.

> Your answer:


## Experiment A — `hidden_dims` (Actor depth/width)

Try **[128]** (shallow), **[256,128]** (baseline), **[256,128,64]**, **[512,256,128]**.


In [ ]:
hidden_configs = {
    '[128]': [128],
    '[256,128] (baseline)': [256, 128],
    '[256,128,64]': [256, 128, 64],
    '[512,256,128]': [512, 256, 128],
}
hidden_results = {}
for name, dims in hidden_configs.items():
    print(f"hidden_dims={name}")
    hidden_results[name] = train_probe_actor(hidden_dims=dims, dropout=0.1, allow_short=False, epochs=40)

plt.figure(figsize=(8,5))
for name, res in hidden_results.items():
    plt.plot(res['val_returns'], label=name)
plt.xlabel('epoch'); plt.ylabel('val mean return'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Effect of Actor hidden_dims on validation return')
plt.show()

pd.DataFrame([{
    'hidden_dims': name, 'final_val_return': r['final_val_return'],
    'sharpe_like': r['sharpe_like'], 'effective_n_assets': r['effective_n_assets'], 'n_params': r['n_params']
} for name, r in hidden_results.items()])


**Your task:** does a deeper/wider Actor actually find a better allocation, or does it just overfit the training window (train return keeps climbing, val return doesn't)? Given only ~630 training samples, is `[512,256,128]` justified?

> Your answer:


## Experiment B — `dropout`

Try **0.0, 0.1 (baseline), 0.3, 0.5**.


In [ ]:
dropout_vals = [0.0, 0.1, 0.3, 0.5]
dropout_results = {}
for p in dropout_vals:
    print(f"dropout={p}")
    dropout_results[p] = train_probe_actor(hidden_dims=[256, 128], dropout=p, allow_short=False, epochs=40)

plt.figure(figsize=(8,5))
for p, res in dropout_results.items():
    plt.plot(res['val_returns'], label=f'dropout={p}')
plt.xlabel('epoch'); plt.ylabel('val mean return'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Effect of Actor dropout on validation return')
plt.show()

pd.DataFrame([{'dropout': p, 'final_val_return': r['final_val_return'], 'sharpe_like': r['sharpe_like']} for p, r in dropout_results.items()])


**Your task:** which dropout value gives the best *risk-adjusted* result (`sharpe_like`), not just raw return? Higher return with wildly higher volatility isn't necessarily better.

> Your answer:


## Experiment C — `allow_short` (long-only vs. long-short)

Long-only (`allow_short=False`) normalizes weights via **softmax** — always positive, sums to 1. Long-short (`allow_short=True`) normalizes via **L1 norm** — can hold negative (short) positions.


In [ ]:
short_results = {}
for allow_short in [False, True]:
    label = 'long-short' if allow_short else 'long-only (baseline)'
    print(label)
    short_results[label] = train_probe_actor(hidden_dims=[256, 128], dropout=0.1, allow_short=allow_short, epochs=40)

fig, axes = plt.subplots(1, 2, figsize=(13,4))
for label, res in short_results.items():
    axes[0].plot(res['val_returns'], label=label)
axes[0].set_title('Validation return'); axes[0].legend(); axes[0].grid(alpha=0.3)

for label, res in short_results.items():
    axes[1].plot(np.cumsum(res['val_return_series']), label=label)
axes[1].plot(np.cumsum(equal_weight_val_returns), label='equal-weight', ls='--', color='grey')
axes[1].set_title('Cumulative validation return'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

pd.DataFrame([{'mode': l, 'final_val_return': r['final_val_return'], 'sharpe_like': r['sharpe_like'],
               'effective_n_assets': r['effective_n_assets']} for l, r in short_results.items()])


**Your task:** does allowing short positions meaningfully improve risk-adjusted return here, or mostly just add instability? For a CSE banking-sector strategy, is long-short even practical (liquidity/borrowing constraints for shorting Sri Lankan banking stocks are real-world considerations beyond what this notebook can capture)?

> Your answer:


## Wrap-up

Log your Day 4 (Actor) choices — you'll reuse them on Day 5 when you add the Critic network and the real Sharpe/EVaR reward.


In [ ]:
day4_choices = {
    'actor_hidden_dims': None,   # e.g. [256, 128]
    'actor_dropout': None,       # e.g. 0.1
    'allow_short': None,         # True | False
}
import json
print(json.dumps(day4_choices, indent=2))
with open('/content/day4_choices.json', 'w') as f:
    json.dump(day4_choices, f, indent=2)


**Next up — Day 5:** Critic Network (`critic.py`) + your real reward function (`reward.py`: Sharpe ratio − transaction costs − EVaR penalty). This replaces today's simplified "maximize raw return" objective with the actual risk-aware reward your research uses, and introduces `V(s)` for advantage estimation. You'll tune Critic depth, the Sharpe-ratio lookback window, the EVaR penalty weight, and the transaction-cost coefficient.
